# Delivery Tutorial: Conditional GANs for TU-Graz Landing

Authors: **Erik Figueiral Alonso** and **Roi Cores Cabaleiro**.

This notebook is the delivery tutorial. It does not duplicate training code: it imports the Python modules used by the command-line runners and shows the workflow from dataset inspection to baseline, improvements, metrics, qualitative comparison, and demo launch.

## 0. Environment

Run this notebook from `/home/erik/cv2-image-to-image` with the `.venv` kernel. The project is Python-only and the same modules are used by the notebook, training scripts, evaluation, and Gradio demo.

In [ ]:
from __future__ import annotations
from pathlib import Path
import csv
import json
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PYTHON = ROOT / '.venv' / 'bin' / 'python'
print('Project root:', ROOT)
print('Python:', PYTHON)
print('Python exists:', PYTHON.exists())

## 1. Dataset Reader

The TU-Graz Landing samples are paired images. In our dataset format the real RGB image is on the left and the semantic label map is on the right. The generator receives the label map and predicts the RGB image.

In [ ]:
from src.dataset import PairedImageDatasetConfig, PairedImageReader, build_datasets_from_config

cfg = PairedImageDatasetConfig(
    data_root=ROOT / 'data' / 'raw',
    split_dir=ROOT / 'data' / 'splits',
    label_side='right',
    image_size=(256, 256),
    split_seed=42,
)
reader = PairedImageReader.from_config(cfg)
print('Total paired files:', len(reader))
datasets = build_datasets_from_config(cfg, use_augmentation=False)
train_ds, val_ds, test_ds = datasets['train'], datasets['val'], datasets['test']
print('Split sizes:', len(train_ds), len(val_ds), len(test_ds))

In [ ]:
import matplotlib.pyplot as plt

sample = test_ds[0]
label = sample['label_map'].permute(1, 2, 0).numpy() * 0.5 + 0.5
target = sample['real_image'].permute(1, 2, 0).numpy() * 0.5 + 0.5
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(label); axes[0].set_title('Input label map'); axes[0].axis('off')
axes[1].imshow(target); axes[1].set_title('Target RGB image'); axes[1].axis('off')
plt.tight_layout()

## 2. Experiment Classes

All variants inherit from a common experiment interface. The notebook shows the commands but the actual implementation lives in `src/experiments.py`, `src/train.py`, `src/train_attention_pix2pix.py`, `src/train_transfer.py`, and `run_experiment.py`.

In [ ]:
from src.experiments import (
    BaselinePix2PixExperiment,
    LSGANExperiment,
    AttentionPix2PixExperiment,
    Pix2PixHDLiteExperiment,
    TransferLearningExperiment,
)

experiments = [
    BaselinePix2PixExperiment(),
    LSGANExperiment(),
    AttentionPix2PixExperiment(),
    Pix2PixHDLiteExperiment(),
    TransferLearningExperiment(),
]
for exp in experiments:
    print(type(exp).__name__)

## 3. Baseline and Improvements

The baseline is Pix2Pix: U-Net generator, PatchGAN discriminator, adversarial loss plus paired reconstruction. The improvements are added one by one: LSGAN, self-attention, Pix2PixHD-lite, and transfer learning.

In [ ]:
training_commands = {
    'baseline': 'python run_experiment.py baseline',
    'lsgan': 'python run_experiment.py lsgan',
    'attention': 'python run_experiment.py attention',
    'pix2pixhd_lite': 'python run_experiment.py pix2pixhd_lite',
    'transfer_learning': 'python -m src.train_transfer --help',
}
for name, cmd in training_commands.items():
    print(f'{name}: {cmd}')

## 4. Current Results Table

The table below reads each `history.csv` and selects the best validation epoch by lowest MAE. The modified 50-epoch experiments will appear automatically after they finish because they follow the same run-folder structure.

In [ ]:
import pandas as pd

run_names = [
    'final_baseline_bce_l1_e40',
    'final_lsgan_l1_100_e40',
    'final_attention_lsgan_e40',
    'final_pix2pixhd_lite_ms2_fm10_e40',
    'final_transfer_resnet18_e50',
    'final_baseline_bce_l50_aug_e50',
    'final_lsgan_l50_aug_e50',
    'final_lsgan_l2_aug_e50',
    'final_attention_lsgan_l50_aug_e50',
    'final_pix2pixhd_lite_ms2_fm5_l75_e50',
]
rows = []
for run in run_names:
    history = ROOT / 'outputs' / 'runs' / run / 'history.csv'
    if not history.exists():
        rows.append({'run': run, 'status': 'pending', 'best_epoch': None, 'val_mae': None, 'val_psnr': None, 'val_ssim': None})
        continue
    df = pd.read_csv(history)
    best = df.loc[df['val_mae'].idxmin()]
    rows.append({
        'run': run,
        'status': 'complete/time-limited',
        'best_epoch': int(best['epoch']),
        'val_mae': float(best['val_mae']),
        'val_psnr': float(best['val_psnr']),
        'val_ssim': float(best['val_ssim']),
    })
results = pd.DataFrame(rows)
results

## 5. Metric Chain

The evaluation pipeline follows a Chain of Responsibility: each handler computes one family of metrics and forwards the same context. This lets the same generated images be evaluated with paired metrics, structural metrics, C2ST, generative precision/recall, and FID when enabled.

In [ ]:
from src.metric_chain import (
    PixelStructureMetricsHandler,
    DistributionMetricsHandler,
    PerceptualMetricsHandler,
)

chain = PixelStructureMetricsHandler()
chain.set_next(DistributionMetricsHandler()).set_next(PerceptualMetricsHandler(include_fid=False, include_lpips=False))
print(chain)

Metric interpretation: MAE is paired fidelity, not diversity. C2ST near 0.5 is good because real and generated images are hard to distinguish. Generative precision and recall separate realism from coverage. FID is valid here only when computed between generated images and real images from the same TU-Graz split.

## 6. Qualitative Comparison

The most honest view is visual: input label map, generated output, and target image. These grids are also used by the LaTeX report.

In [ ]:
from IPython.display import Image, display

figure_paths = [
    ROOT / 'report/figures/final_baseline_bce_l1_e40_grid.png',
    ROOT / 'report/figures/final_lsgan_l1_100_e40_grid.png',
    ROOT / 'report/figures/final_attention_lsgan_e40_grid.png',
    ROOT / 'report/figures/final_pix2pixhd_lite_ms2_fm10_e40_grid.png',
]
for path in figure_paths:
    if path.exists():
        print(path.name)
        display(Image(filename=str(path)))
    else:
        print('Missing:', path)

## 7. Demo

The Gradio demo auto-discovers checkpoints in `models/` and `outputs/runs/final_*`, so it no longer needs a long list of command-line checkpoint arguments.

In [ ]:
print('Launch demo from a terminal:')
print('cd /home/erik/cv2-image-to-image')
print('source .venv/bin/activate')
print('python demo/visual_demo.py --host 0.0.0.0 --port 7860')
print('Open http://localhost:7860')
print('Use input mode paired-right for original TU-Graz paired images.')

## 8. Delivery Checklist

- Python code: `src/`, `run_experiment.py`, `demo/`.
- Notebook tutorial: this notebook.
- Visual demo: `demo/visual_demo.py`.
- Report: `report/main.tex` with IEEE style.
- Checkpoints: each run has its own `outputs/runs/<run_name>/checkpoints/best_generator.pt`.
- Pending updates: after the modified 50-epoch queue finishes, rerun the results-table cell and update the final LaTeX table.